# Neo4j Query Notebook

This notebook demonstrates how to:
- Connect to Neo4j using environment variables
- Test the connection
- Execute complex Cypher queries
- Analyze and visualize results


## 1. Imports and Setup


In [31]:
import os
import sys
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path to import neo4jClient
sys.path.append(os.path.dirname(os.path.abspath('.')))

from neo4jClient import Neo4jClient

print("✓ Imports successful")


✓ Imports successful


## 2. Load Environment Variables


## 2.5. Verify .env File Contents


In [32]:
# Verify .env file exists and show its contents (with password masked)
import os
from pathlib import Path

# Find .env file
env_paths_to_check = [
    Path('.env'),  # Current directory
    Path('../.env'),  # Parent directory
    Path('../../.env'),  # Two levels up (project root)
]

env_file_found = None
for env_path in env_paths_to_check:
    if env_path.exists():
        env_file_found = env_path
        break

if env_file_found:
    print(f"✓ Found .env file at: {env_file_found.absolute()}\n")
    print("=" * 60)
    print(".env File Contents (password masked):")
    print("=" * 60)
    
    with open(env_file_found, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.rstrip()
            # Mask password values
            if 'PASSWORD' in line.upper() and '=' in line:
                key, value = line.split('=', 1)
                masked_value = '*' * len(value) if value else 'EMPTY'
                print(f"{line_num:3d}: {key}={masked_value}")
            else:
                print(f"{line_num:3d}: {line}")
    print("=" * 60)
else:
    print("✗ No .env file found in:")
    for path in env_paths_to_check:
        print(f"  - {path.absolute()}")
    print("\nPlease create a .env file with:")
    print("NEO4J_URI=bolt://localhost:7687")
    print("NEO4J_USER=neo4j")
    print("NEO4J_PASSWORD=your_password")
    print("NEO4J_DATABASE=neo4j")


✓ Found .env file at: c:\Users\tomas\JavaProjects\Aibeceles\ml\neo4j\..\.env

.env File Contents (password masked):
  1:    NEO4J_URI=bolt://localhost:7687
  2:    NEO4J_USER=neo4j
  3:    NEO4J_PASSWORD=***********
  4:    NEO4J_DATABASE=d4seed1


In [35]:
# Load environment variables from .env file
load_dotenv(override=True)

# Get connection parameters
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# Display configuration (masked password)
print("Connection Configuration:")
print(f"  URI: {NEO4J_URI}")
print(f"  User: {NEO4J_USER}")
print(f"  Password: {'*' * len(NEO4J_PASSWORD) if NEO4J_PASSWORD else 'NOT SET'}")
print(f"  Database: {NEO4J_DATABASE}")


Connection Configuration:
  URI: bolt://localhost:7687
  User: neo4j
  Password: ***********
  Database: d4seed1


## 3. Initialize Connection


In [36]:
try:
    client = Neo4jClient(
        uri=NEO4J_URI,
        user=NEO4J_USER,
        password=NEO4J_PASSWORD
    )
    print("✓ Neo4j client initialized successfully")
except ConnectionError as e:
    print(f"✗ Connection failed: {e}")
    raise


✓ Neo4j client initialized successfully


## 3.5. Check Available Databases (Troubleshooting)


In [37]:
# Check what databases are available in your Neo4j instance
list_databases_query = "SHOW DATABASES"

try:
    databases_df = client.run_query(list_databases_query, "system")
    print("Available databases in Neo4j:")
    display(databases_df)
    
    # Show only database names for clarity
    if 'name' in databases_df.columns:
        print("\nDatabase names:")
        for db_name in databases_df['name']:
            print(f"  - {db_name}")
except RuntimeError as e:
    print(f"Could not list databases: {e}")
    print("\nTip: If using Neo4j Community Edition, you may only have 'neo4j' database.")
    print("Update your .env file to set: NEO4J_DATABASE=neo4j")


Available databases in Neo4j:


,name,type,aliases,access,address,role,writer,requestedStatus,currentStatus,statusMessage,default,home,constituents
0,d4seed1,standard,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]
1,d5seed1,standard,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]
2,d6seed1,standard,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]
3,d7seed1,standard,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]
4,d8seed1,standard,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]
5,neo4j,standard,[],read-write,localhost:7687,primary,True,online,online,,True,True,[]
6,system,system,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]
7,tagtest,standard,[],read-write,localhost:7687,primary,True,online,online,,False,False,[]



Database names:
  - d4seed1
  - d5seed1
  - d6seed1
  - d7seed1
  - d8seed1
  - neo4j
  - system
  - tagtest


## 3.6. Override Database Name (If Needed)

If the database from .env doesn't exist, you can override it here.


In [38]:
# OPTIONAL: Override the database name if needed
# Uncomment and modify the line below to use a different database
#NEO4J_DATABASE = "d4seed1"  # Most common default database name

print(f"Current database setting: {NEO4J_DATABASE}")
print("\nIf this database doesn't exist, uncomment the line above and set it to one of the available databases.")


Current database setting: d4seed1

If this database doesn't exist, uncomment the line above and set it to one of the available databases.


## 4. Connection Test


In [39]:
# Run a simple test query
test_query = "MATCH (n) RETURN count(n) AS total_nodes"

try:
    result_df = client.run_query(test_query, NEO4J_DATABASE)
    print("✓ Connection test successful!")
    print(f"\nDatabase contains {result_df['total_nodes'].iloc[0]} total nodes")
    display(result_df)
except RuntimeError as e:
    print(f"✗ Query failed: {e}")


✓ Connection test successful!

Database contains 1170907 total nodes


,total_nodes
0,1170907


## 5. Run Complex Query

Execute your UNION query to find CreatedBy nodes with specific pArray values and their relationships.


In [41]:
# Your complex query
query = """
CALL () {
    MATCH (n:CreatedBy {pArray:"[7, 16, 17, 10, 0]"})
    WITH n LIMIT 20
    MATCH (n)<-[r:CreatedBye]-(m)
    RETURN n, r, m
    
    UNION
    
    MATCH (n:CreatedBy {pArray:"[7, 15, 17, 10, 0]"})
    WITH n LIMIT 20
    MATCH (n)<-[r:CreatedBye]-(m)
    RETURN n, r, m
}
RETURN n, r, m
"""

try:
    results_df = client.run_query(query, NEO4J_DATABASE)
    print(f"✓ Query executed successfully!")
    print(f"  Returned {len(results_df)} rows")
    display(results_df)
except RuntimeError as e:
    print(f"✗ Query execution failed: {e}")
    print("\nPlease check:")
    print("  - Node labels (CreatedBy) exist in database")
    print("  - Relationship type (CreatedBye) is correct")
    print("  - Property 'pArray' exists on nodes")


✓ Query executed successfully!
  Returned 10 rows


,n,r,m
0,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 1.0000000000229647, -46.0...","{'vmResult': '[0.0, 1.0000000000229647, -46.00..."
1,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 4.000000000000114, -132.0...","{'vmResult': '[0.0, 4.000000000000114, -132.0,..."
2,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 12.0, -252.0, 816.0]', 'm...","{'vmResult': '[0.0, 12.0, -252.0, 816.0]', 'mu..."
3,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 24.0, -240.0]', 'muList':...","{'vmResult': '[0.0, 24.0, -240.0]', 'muList': ..."
4,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 24.0]', 'muList': '[]', '...","{'vmResult': '[0.0, 24.0]', 'muList': '[]', 'd..."
5,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 1.0000000000184173, -46.0...","{'vmResult': '[0.0, 1.0000000000184173, -46.00..."
6,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 4.000000000000057, -132.0...","{'vmResult': '[0.0, 4.000000000000057, -132.0,..."
7,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 12.0, -252.0, 816.0]', 'm...","{'vmResult': '[0.0, 12.0, -252.0, 816.0]', 'mu..."
8,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 24.0, -240.0]', 'muList':...","{'vmResult': '[0.0, 24.0, -240.0]', 'muList': ..."
9,"{'skipList': '[false, false, false, false]', '...","({'vmResult': '[0.0, 24.0]', 'muList': '[]', '...","{'vmResult': '[0.0, 24.0]', 'muList': '[]', 'd..."


## 6. Data Analysis

Explore the returned data to understand the structure and content.


In [ ]:
# Display DataFrame info
print("DataFrame Information:")
print(f"  Shape: {results_df.shape}")
print(f"  Columns: {list(results_df.columns)}")
print(f"\nData Types:")
print(results_df.dtypes)
print(f"\nFirst few rows:")
display(results_df.head())


## 7. Extract and Analyze Node/Relationship Properties

Parse the Neo4j objects to extract useful information.


In [ ]:
# Example: Extract properties from nodes
if len(results_df) > 0:
    print("Sample node (n) properties:")
    sample_n = results_df['n'].iloc[0]
    print(f"  Type: {type(sample_n)}")
    if hasattr(sample_n, 'items'):
        for key, value in sample_n.items():
            print(f"  {key}: {value}")
    
    print("\nSample relationship (r) properties:")
    sample_r = results_df['r'].iloc[0]
    print(f"  Type: {type(sample_r)}")
    if hasattr(sample_r, 'items'):
        for key, value in sample_r.items():
            print(f"  {key}: {value}")
    
    print("\nSample connected node (m) properties:")
    sample_m = results_df['m'].iloc[0]
    print(f"  Type: {type(sample_m)}")
    if hasattr(sample_m, 'items'):
        for key, value in sample_m.items():
            print(f"  {key}: {value}")
else:
    print("No results returned from query")


## 8. Optional: Run Additional Queries

Add more cells below to run additional queries as needed.


In [ ]:
# Example: Count nodes by pArray value
count_query = """
MATCH (n:CreatedBy)
WHERE n.pArray IN ["[7, 16, 17, 10, 0]", "[7, 15, 17, 10, 0]"]
RETURN n.pArray AS pArray, count(n) AS count
ORDER BY count DESC
"""

try:
    count_df = client.run_query(count_query, NEO4J_DATABASE)
    print("Node counts by pArray:")
    display(count_df)
except RuntimeError as e:
    print(f"Query failed: {e}")


## 9. Cleanup

Close the Neo4j connection when done.


In [ ]:
# Close the connection
client.close()
print("✓ Connection closed successfully")


In [34]:
import os
print("System environment variable:")
print(os.environ.get('NEO4J_DATABASE', 'NOT SET IN SYSTEM'))

System environment variable:
d4seed1$
